In [0]:
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("load_type","")
dbutils.widgets.text("merge_keys","")

landing_timestamp=dbutils.widgets.get("landing_timestamp")
table_name=dbutils.widgets.get("table_name")
merge_keys=dbutils.widgets.get("merge_keys")
load_type=dbutils.widgets.get("load_type").upper()


In [0]:
bronze_df=spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp='{landing_timestamp}'")


In [0]:
from pyspark.sql import functions as f
updated_df=bronze_df.withColumn("insert_timestamp",f.date_format(f.current_timestamp(),"yyyy-MM-dd HH:mm:ss")).withColumn("update_timestamp",f.date_format(f.current_timestamp(),"yyyy-MM-dd HH:mm:ss"))

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as f

if load_type == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(f"adb_caresync.silver.{table_name}")
elif load_type == "APPEND":
    updated_df.write.mode("append").saveAsTable(f"adb_caresync.silver.{table_name}")
elif load_type=="MERGE":
    if not merge_keys or merge_keys.strip() == "":
        raise ValueError("merge_keys cannot be empty when load_type is MERGE")
    if spark.catalog.tableExists(f"adb_caresync.silver.{table_name}"):
        merge_condition = " AND ".join([f"t.{key.strip()} = s.{key.strip()}" for key in merge_keys.split(",")])
        target = DeltaTable.forName(spark, f"adb_caresync.silver.{table_name}")
        update_cols = {col: f"s.{col}" for col in updated_df.columns if col != "insert_timestamp"}
        target.alias("t").merge(
            updated_df.alias("s"),
            merge_condition
        ).whenMatchedUpdate(
            set=update_cols
        ).whenNotMatchedInsertAll(
        ).execute()
    else:
        updated_df.write.mode("overwrite").saveAsTable(f"adb_caresync.silver.{table_name}")
else:
    raise ValueError("load_type must be FULL or APPEND or MERGE")

In [0]:
records_written = updated_df.count()
dbutils.notebook.exit(str(records_written))